# Orientation-Specific Four-Stage Hierarchy CNN / BiGRU Experiment

Structure:

1. **Target model**: normal augmented CNN on all sequences.
2. **Orientation model**: normal augmented CNN on target-only sequences.
3. **Gesture action models**: four orientation-specific CNN + `bigru` / `attention` models.
4. **Gesture position models**: four orientation-specific CNN + `bigru` / `attention` models.

There are **10 models total**:

```text
1 target model
1 orientation model
4 orientation-specific gesture_action models
4 orientation-specific gesture_position models
```

Each model has its own:

```text
run mode: search / best_params / model
search mode: grid / bayesian
param space
saved model path
```

Grid defaults are intentionally **one combination only** per model.


In [ ]:
from __future__ import annotations

import os
import json
import joblib
from pathlib import Path
from datetime import datetime
from types import SimpleNamespace

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, GridSearchCV
from sklearn.metrics import f1_score, classification_report, confusion_matrix

try:
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
except Exception as exc:
    BayesSearchCV = None
    Categorical = None
    Integer = None
    Real = None
    print("BayesSearchCV unavailable. Use grid search.", exc)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

In [ ]:
# ============================================================
# Import paths
# ============================================================

import sys

# Change only this when your uploaded Kaggle dataset name changes.
# Example below gives: /kaggle/input/datasets/keithmarange/fourstage/
fourstage_dataset_name = "fourstage"

sys.path.append(f"/kaggle/input/datasets/keithmarange/{fourstage_dataset_name}/")
sys.path.append("/kaggle/input/cmi-competition-code")

# Local fallback for when running outside Kaggle after downloading files.
sys.path.append(".")

import data_utils
import utils_hierarchy_position_aug as utils
import orientation_specific_hierarchy_utils as hs_utils

print("fourstage_dataset_name:", fourstage_dataset_name)

In [ ]:
# ============================================================
# Config
# ============================================================

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
results_dir = Path("results")
results_dir.mkdir(parents=True, exist_ok=True)

random_state = 42
n_cv_splits = 2
model_verbose = 3
search_verbose = 3

# Data size controls
use_subject_holdout = False
holdout_size = 0.2
use_train_subset = True
train_sequence_frac = 0.2

# Search controls
n_iter_default = 25
error_score = np.nan

corrector_name = "sensor_corrector"
pipe_name = "sequence_builder"
classifier_name = "classifier"

# Model run modes:
# "search"      -> run grid / Bayes
# "best_params" -> load params from previous_best_params_path, then refit
# "model"       -> load fitted joblib pipeline, no refit
model_run_mode = {
    "target": "search",
    "orientation": "search",
    "action_lie_back": "search",
    "action_lie_side_non_dom": "search",
    "action_seated_lean_face_down": "search",
    "action_seated_straight": "search",
    "position_lie_back": "search",
    "position_lie_side_non_dom": "search",
    "position_seated_lean_face_down": "search",
    "position_seated_straight": "search",
}

# Each model can independently use grid or Bayesian search.
# Grid spaces below are one-combination defaults.
model_search_mode = {
    "target": "grid",
    "orientation": "grid",
    "action_lie_back": "grid",
    "action_lie_side_non_dom": "grid",
    "action_seated_lean_face_down": "grid",
    "action_seated_straight": "grid",
    "position_lie_back": "grid",
    "position_lie_side_non_dom": "grid",
    "position_seated_lean_face_down": "grid",
    "position_seated_straight": "grid",
}

model_n_iter = {k: n_iter_default for k in model_run_mode}

# Optional previous params / model loading
previous_best_params_path = None  # e.g. "results/orientation_specific_best_params_20260521_1234.json"

previous_model_paths = {k: None for k in model_run_mode}

print("timestamp:", timestamp)
print("model_run_mode:", model_run_mode)
print("model_search_mode:", model_search_mode)

In [ ]:
# ============================================================
# Load data
# ============================================================

data_root = data_utils.find_data_root()

raw_train_df = pd.read_csv(data_root / "train.csv")
raw_test_df = pd.read_csv(data_root / "test.csv")
train_demo_df = pd.read_csv(data_root / "train_demographics.csv")
test_demo_df = pd.read_csv(data_root / "test_demographics.csv")

print("Using Kaggle data folder:", data_root)
print("raw_train_df:", raw_train_df.shape)
print("raw_test_df:", raw_test_df.shape)
print("train_demo_df:", train_demo_df.shape)
print("test_demo_df:", test_demo_df.shape)
print("train sequences:", raw_train_df["sequence_id"].nunique())
print("subjects:", raw_train_df["subject"].nunique())

In [ ]:
# ============================================================
# Base dataframe + helper targets
# ============================================================

train_df = raw_train_df.set_index("row_id").copy(deep=True)

train_df.loc[:, "gesture_position"] = train_df["gesture"].str.split(" - ").str[0]
train_df.loc[:, "gesture_action"] = train_df["gesture"].str.split(" - ").str[-1]
train_df.loc[:, "is_target"] = train_df["sequence_type"].eq("Target").astype(int)
train_df.loc[:, "hierarchical_gesture"] = np.where(
    train_df["sequence_type"].eq("Target"),
    train_df["gesture"],
    "Non-Target",
)

print("all sequences:", train_df["sequence_id"].nunique())
print("target sequences:", train_df.loc[train_df["sequence_type"].eq("Target"), "sequence_id"].nunique())
print("gesture_action classes:", sorted(train_df.loc[train_df["sequence_type"].eq("Target"), "gesture_action"].dropna().unique()))
print("orientation classes:", sorted(train_df.loc[train_df["sequence_type"].eq("Target"), "orientation"].dropna().unique()))
print("gesture_position classes:", sorted(train_df.loc[train_df["sequence_type"].eq("Target"), "gesture_position"].dropna().unique()))
print("full gesture classes:", sorted(train_df.loc[train_df["sequence_type"].eq("Target"), "gesture"].dropna().unique()))

In [ ]:
# ============================================================
# Subject holdout split + optional train subset
# ============================================================

seq_meta = (
    train_df
    .drop_duplicates("sequence_id")
    [["sequence_id", "subject", "sequence_type", "gesture", "is_target", "hierarchical_gesture"]]
    .reset_index(drop=True)
)

if use_subject_holdout:
    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=holdout_size,
        random_state=random_state,
    )
    train_seq_idx, holdout_seq_idx = next(
        splitter.split(
            seq_meta,
            y=seq_meta["is_target"],
            groups=seq_meta["subject"],
        )
    )
    train_seq_ids = seq_meta.loc[train_seq_idx, "sequence_id"]
    holdout_seq_ids = seq_meta.loc[holdout_seq_idx, "sequence_id"]
else:
    # Fast debug split by sequence, not subject. Do not use for final generalisation claims.
    train_seq_ids = seq_meta.sample(frac=holdout_size, random_state=random_state)["sequence_id"]
    remaining_seq_ids = seq_meta.loc[~seq_meta["sequence_id"].isin(train_seq_ids), "sequence_id"]
    holdout_seq_ids = remaining_seq_ids.sample(n=len(train_seq_ids), random_state=random_state)

train_model_df = train_df.loc[train_df["sequence_id"].isin(train_seq_ids)].copy()
holdout_df = train_df.loc[train_df["sequence_id"].isin(holdout_seq_ids)].copy()

if use_train_subset:
    train_seq_meta = (
        train_model_df
        .drop_duplicates("sequence_id")
        [["sequence_id", "subject", "sequence_type", "gesture", "is_target"]]
        .reset_index(drop=True)
    )
    sampled_train_seq_ids = (
        train_seq_meta
        .groupby("sequence_type", group_keys=False)
        .sample(frac=train_sequence_frac, random_state=random_state)["sequence_id"]
    )
    train_model_df = train_model_df.loc[train_model_df["sequence_id"].isin(sampled_train_seq_ids)].copy()

target_only_train_df = train_model_df.loc[train_model_df["sequence_type"].eq("Target")].copy()
target_only_holdout_df = holdout_df.loc[holdout_df["sequence_type"].eq("Target")].copy()

print("full sequences:", train_df["sequence_id"].nunique())
print("train sequences:", train_model_df["sequence_id"].nunique())
print("holdout sequences:", holdout_df["sequence_id"].nunique())
print("target-only train sequences:", target_only_train_df["sequence_id"].nunique())
print("target-only holdout sequences:", target_only_holdout_df["sequence_id"].nunique())
print("train subjects:", train_model_df["subject"].nunique())
print("holdout subjects:", holdout_df["subject"].nunique())
print("subject overlap:", len(set(train_model_df["subject"]) & set(holdout_df["subject"])))

print("\ntrain sequence_type counts:")
print(train_model_df.drop_duplicates("sequence_id")["sequence_type"].value_counts())

print("\nholdout sequence_type counts:")
print(holdout_df.drop_duplicates("sequence_id")["sequence_type"].value_counts())

In [ ]:
# ============================================================
# Orientation key mapping + per-orientation training data
# ============================================================

orientation_key_map = {
    "Lie on Back": "lie_back",
    "Lie on Side - Non Dominant": "lie_side_non_dom",
    "Seated Lean Non Dom - FACE DOWN": "seated_lean_face_down",
    "Seated Straight": "seated_straight",
}

orientation_value_by_key = {v: k for k, v in orientation_key_map.items()}

model_keys = [
    "target",
    "orientation",
    "action_lie_back",
    "action_lie_side_non_dom",
    "action_seated_lean_face_down",
    "action_seated_straight",
    "position_lie_back",
    "position_lie_side_non_dom",
    "position_seated_lean_face_down",
    "position_seated_straight",
]

model_target_by_key = {
    "target": "is_target",
    "orientation": "orientation",
    "action_lie_back": "gesture_action",
    "action_lie_side_non_dom": "gesture_action",
    "action_seated_lean_face_down": "gesture_action",
    "action_seated_straight": "gesture_action",
    "position_lie_back": "gesture_position",
    "position_lie_side_non_dom": "gesture_position",
    "position_seated_lean_face_down": "gesture_position",
    "position_seated_straight": "gesture_position",
}

model_family_by_key = {
    "target": "cnn",
    "orientation": "cnn",
    "action_lie_back": "multibranch",
    "action_lie_side_non_dom": "multibranch",
    "action_seated_lean_face_down": "multibranch",
    "action_seated_straight": "multibranch",
    "position_lie_back": "multibranch",
    "position_lie_side_non_dom": "multibranch",
    "position_seated_lean_face_down": "multibranch",
    "position_seated_straight": "multibranch",
}

orientation_filter_by_key = {
    "target": None,
    "orientation": None,
    "action_lie_back": "Lie on Back",
    "action_lie_side_non_dom": "Lie on Side - Non Dominant",
    "action_seated_lean_face_down": "Seated Lean Non Dom - FACE DOWN",
    "action_seated_straight": "Seated Straight",
    "position_lie_back": "Lie on Back",
    "position_lie_side_non_dom": "Lie on Side - Non Dominant",
    "position_seated_lean_face_down": "Seated Lean Non Dom - FACE DOWN",
    "position_seated_straight": "Seated Straight",
}

# Fallbacks used if an orientation-specific model is unavailable or all fits fail.
fallback_action_by_orientation = (
    target_only_train_df
    .drop_duplicates("sequence_id")
    .groupby("orientation")["gesture_action"]
    .agg(lambda s: s.mode().iloc[0])
    .to_dict()
)

fallback_position_by_orientation = (
    target_only_train_df
    .drop_duplicates("sequence_id")
    .groupby("orientation")["gesture_position"]
    .agg(lambda s: s.mode().iloc[0])
    .to_dict()
)

fallback_target_gesture = target_only_train_df.drop_duplicates("sequence_id")["gesture"].mode().iloc[0]
print("fallback_target_gesture:", fallback_target_gesture)
print("fallback_action_by_orientation:", fallback_action_by_orientation)
print("fallback_position_by_orientation:", fallback_position_by_orientation)

In [ ]:
# ============================================================
# Previous best params loading
# ============================================================

previous_best_params = {k: None for k in model_keys}

if previous_best_params_path is not None:
    with open(previous_best_params_path, "r") as f:
        previous_best_params = json.load(f)
    print("Loaded previous best params:", previous_best_params_path)
else:
    print("No previous best params path supplied. Search modes will run normally unless model_run_mode uses 'model'.")

In [ ]:
# ============================================================
# Grid param spaces — one default combination per model
# ============================================================

common_seq_grid = {
    f"{pipe_name}__acc_modes": ["smoothed|velocity|displacement|jerk"],
    f"{pipe_name}__rotation_modes": ["quaternion|rot6d|angular_velocity"],
    f"{pipe_name}__sampling_rate": [10],
    f"{pipe_name}__interp_mode": ["linear"],
    f"{pipe_name}__standardize": ["mean_std"],
    f"{pipe_name}__linear_acc_mode": ["baseline"],
    f"{pipe_name}__use_acc_magnitude": [True],
    f"{pipe_name}__use_linear_acc_magnitude": [True],
    f"{pipe_name}__tof_mode": ["pooled_stats"],
    f"{pipe_name}__tof_fill_mode": ["far_255"],
    f"{pipe_name}__thm_mode": ["centered"],
    f"{pipe_name}__motion_filter_mode": ["kalman"],
    f"{pipe_name}__kalman_process_noise": [1e-3],
    f"{pipe_name}__kalman_measurement_noise": [1e-1],
    f"{pipe_name}__use_dead_reckoning": [False],
    f"{pipe_name}__clip_value": [50.0],
    f"{pipe_name}__window_size": [30],
    f"{pipe_name}__smooth_alpha": [0.8],
}

cnn_grid = {
    f"{classifier_name}__maxlen": [160],
    f"{classifier_name}__conv_filters": ["128-256-256"],
    f"{classifier_name}__kernel_sizes": ["5-5-5"],
    f"{classifier_name}__pool_sizes": ["none-none-none"],
    f"{classifier_name}__use_batch_norm": [True],
    f"{classifier_name}__spatial_dropout": [0.25],
    f"{classifier_name}__dense_units": ["64"],
    f"{classifier_name}__dropout": [0.45],
    f"{classifier_name}__learning_rate": [5e-5],
    f"{classifier_name}__batch_size": [32],
    f"{classifier_name}__epochs": [50],
    f"{classifier_name}__patience": [5],
    f"{classifier_name}__use_mixup": [False],
    f"{classifier_name}__mixup_alpha": [0.4],
    f"{classifier_name}__mixup_size": [1.0],
    f"{classifier_name}__use_gaussian_noise": [False],
    f"{classifier_name}__noise_std": [0.01],
    f"{classifier_name}__use_time_mask": [False],
    f"{classifier_name}__time_mask_ratio": [0.1],
    f"{classifier_name}__use_time_shift": [False],
    f"{classifier_name}__max_shift_pct": [0.25],
    f"{classifier_name}__use_time_stretch": [False],
    f"{classifier_name}__time_stretch_min_rate": [0.8],
    f"{classifier_name}__time_stretch_max_rate": [1.2],
    f"{classifier_name}__use_magnitude_scaling": [False],
    f"{classifier_name}__magnitude_scale_min": [0.9],
    f"{classifier_name}__magnitude_scale_max": [1.1],
    f"{classifier_name}__use_channel_dropout": [False],
    f"{classifier_name}__channel_dropout_prob": [0.0],
    f"{classifier_name}__use_modality_dropout": [False],
    f"{classifier_name}__drop_acc_prob": [0.0],
    f"{classifier_name}__drop_rot_prob": [0.0],
    f"{classifier_name}__drop_tof_prob": [0.0],
    f"{classifier_name}__drop_thm_prob": [0.0],
}

multibranch_grid = {
    f"{classifier_name}__maxlen": [160],
    f"{classifier_name}__branch_filters": [
        {"acc": "128-256-512", "rot": "128-256", "tof": "64", "thm": "16"},
    ],
    f"{classifier_name}__branch_kernel_sizes": [
        {"acc": "5-5-5", "rot": "3-3", "tof": "3", "thm": "3"},
    ],
    f"{classifier_name}__branch_pool_sizes": [
        {"acc": "none", "rot": "none", "tof": "none", "thm": "none"},
    ],
    f"{classifier_name}__fusion_mode": ["bigru"],
    f"{classifier_name}__attention_heads": [8],
    f"{classifier_name}__gru_units": [192],
    f"{classifier_name}__use_batch_norm": [True],
    f"{classifier_name}__spatial_dropout": [0.2],
    f"{classifier_name}__dense_units": ["128-64"],
    f"{classifier_name}__dropout": [0.45],
    f"{classifier_name}__learning_rate": [5e-5],
    f"{classifier_name}__batch_size": [32],
    f"{classifier_name}__epochs": [50],
    f"{classifier_name}__patience": [5],
}

grid_param_spaces = {}

for k in model_keys:
    if model_family_by_key[k] == "cnn":
        grid_param_spaces[k] = {**common_seq_grid, **cnn_grid}
    else:
        grid_param_spaces[k] = {**common_seq_grid, **multibranch_grid}

# Per-model tweaks while still keeping exactly one grid combination per model.
grid_param_spaces["target"][f"{pipe_name}__acc_modes"] = ["smoothed|velocity|displacement|jerk"]
grid_param_spaces["orientation"][f"{pipe_name}__acc_modes"] = ["velocity|displacement"]
grid_param_spaces["orientation"][f"{pipe_name}__rotation_modes"] = ["quaternion|rot6d"]

for k in ["action_lie_back", "action_lie_side_non_dom", "action_seated_lean_face_down", "action_seated_straight"]:
    grid_param_spaces[k][f"{pipe_name}__acc_modes"] = ["smoothed|velocity|displacement|jerk"]
    grid_param_spaces[k][f"{pipe_name}__rotation_modes"] = ["quaternion|rot6d|angular_velocity"]
    grid_param_spaces[k][f"{classifier_name}__fusion_mode"] = ["bigru"]

for k in ["position_lie_back", "position_lie_side_non_dom", "position_seated_lean_face_down", "position_seated_straight"]:
    grid_param_spaces[k][f"{pipe_name}__acc_modes"] = ["raw|velocity|displacement"]
    grid_param_spaces[k][f"{pipe_name}__rotation_modes"] = ["quaternion|rot6d"]
    grid_param_spaces[k][f"{classifier_name}__fusion_mode"] = ["attention"]

print("Grid spaces ready:", list(grid_param_spaces))

In [ ]:
# ============================================================
# Bayesian param spaces — each model has its own search space
# ============================================================

bayes_param_spaces = {}

if Categorical is not None:
    common_seq_bayes = {
        f"{pipe_name}__acc_modes": Categorical([
            "raw|velocity|displacement",
            "smoothed|velocity|displacement|jerk",
        ]),
        f"{pipe_name}__rotation_modes": Categorical([
            "quaternion|rot6d",
            "quaternion|rot6d|angular_velocity",
            "rot6d|angular_velocity",
        ]),
        f"{pipe_name}__sampling_rate": Categorical([10, 20]),
        f"{pipe_name}__interp_mode": Categorical(["linear", "ffill"]),
        f"{pipe_name}__standardize": Categorical(["mean_std"]),
        f"{pipe_name}__linear_acc_mode": Categorical(["baseline"]),
        f"{pipe_name}__use_acc_magnitude": Categorical([True]),
        f"{pipe_name}__use_linear_acc_magnitude": Categorical([True]),
        f"{pipe_name}__tof_mode": Categorical(["pooled_stats", "sensor_stats", "pooled_diff"]),
        f"{pipe_name}__tof_fill_mode": Categorical(["nan_interpolate", "far_255", "zero"]),
        f"{pipe_name}__thm_mode": Categorical(["centered", "diff", "centered_diff"]),
        f"{pipe_name}__motion_filter_mode": Categorical([None, "kalman", "extended_kalman"]),
        f"{pipe_name}__kalman_process_noise": Real(1e-5, 1e-1, prior="log-uniform"),
        f"{pipe_name}__kalman_measurement_noise": Real(1e-4, 1.0, prior="log-uniform"),
        f"{pipe_name}__use_dead_reckoning": Categorical([False, True]),
        f"{pipe_name}__clip_value": Categorical([50.0]),
        f"{pipe_name}__window_size": Integer(8, 90),
        f"{pipe_name}__smooth_alpha": Categorical([None, 0.8]),
    }

    cnn_bayes = {
        f"{classifier_name}__maxlen": Categorical([96, 132, 160, 198]),
        f"{classifier_name}__conv_filters": Categorical(["64-128-256", "128-256-256", "128-256-512"]),
        f"{classifier_name}__kernel_sizes": Categorical(["3-5-5", "5-5-5", "7-5-3"]),
        f"{classifier_name}__pool_sizes": Categorical(["none-none-none"]),
        f"{classifier_name}__use_batch_norm": Categorical([True]),
        f"{classifier_name}__spatial_dropout": Real(0.1, 0.4),
        f"{classifier_name}__dense_units": Categorical(["64", "128", "128-64"]),
        f"{classifier_name}__dropout": Real(0.3, 0.6),
        f"{classifier_name}__learning_rate": Real(1e-5, 3e-4, prior="log-uniform"),
        f"{classifier_name}__batch_size": Categorical([16, 32]),
        f"{classifier_name}__epochs": Categorical([120]),
        f"{classifier_name}__patience": Categorical([20]),
        f"{classifier_name}__use_mixup": Categorical([False, True]),
        f"{classifier_name}__mixup_alpha": Real(0.2, 0.6),
        f"{classifier_name}__mixup_size": Categorical([1.0]),
        f"{classifier_name}__use_gaussian_noise": Categorical([False]),
        f"{classifier_name}__use_time_mask": Categorical([False]),
        f"{classifier_name}__use_time_shift": Categorical([False]),
        f"{classifier_name}__use_time_stretch": Categorical([False]),
        f"{classifier_name}__use_magnitude_scaling": Categorical([False]),
        f"{classifier_name}__use_channel_dropout": Categorical([False]),
        f"{classifier_name}__use_modality_dropout": Categorical([False]),
    }

    multibranch_bayes = {
        f"{classifier_name}__maxlen": Categorical([96, 115, 132, 160, 198]),
        f"{classifier_name}__branch_filters": [
            {"acc": "64-128-256-256", "rot": "64-128", "tof": "64", "thm": "16"},
            {"acc": "128-256-256-256", "rot": "128-128", "tof": "128", "thm": "32"},
            {"acc": "128-256-512", "rot": "128-256", "tof": "64", "thm": "16"},
        ],
        f"{classifier_name}__branch_kernel_sizes": [
            {"acc": "3-5-5", "rot": "3-3", "tof": "3", "thm": "3"},
            {"acc": "5-5-5", "rot": "3-3", "tof": "3", "thm": "3"},
            {"acc": "7-5-3", "rot": "3-3", "tof": "3", "thm": "3"},
        ],
        f"{classifier_name}__branch_pool_sizes": [
            {"acc": "none", "rot": "none", "tof": "none", "thm": "none"},
        ],
        f"{classifier_name}__fusion_mode": Categorical(["bigru", "attention"]),
        f"{classifier_name}__attention_heads": Integer(2, 14),
        f"{classifier_name}__gru_units": Integer(96, 384),
        f"{classifier_name}__use_batch_norm": Categorical([True]),
        f"{classifier_name}__spatial_dropout": Real(0.05, 0.35),
        f"{classifier_name}__dense_units": Categorical(["32", "64", "128", "64-32", "128-64"]),
        f"{classifier_name}__dropout": Real(0.25, 0.6),
        f"{classifier_name}__learning_rate": Real(1e-5, 3e-4, prior="log-uniform"),
        f"{classifier_name}__batch_size": Categorical([16, 32]),
        f"{classifier_name}__epochs": Categorical([120]),
        f"{classifier_name}__patience": Categorical([20]),
    }

    for k in model_keys:
        if model_family_by_key[k] == "cnn":
            bayes_param_spaces[k] = {**common_seq_bayes, **cnn_bayes}
        else:
            raw_space = {**common_seq_bayes, **multibranch_bayes}
            bayes_param_spaces[k] = hs_utils.prepare_branch_param_space(
                raw_space,
                "bayesian",
                Categorical=Categorical,
            )

    # Slight per-model targeting.
    bayes_param_spaces["orientation"][f"{pipe_name}__acc_modes"] = Categorical(["velocity|displacement", "smoothed|velocity|displacement|jerk"])
    bayes_param_spaces["orientation"][f"{pipe_name}__rotation_modes"] = Categorical(["quaternion|rot6d", "quaternion|rot6d|angular_velocity"])

    for k in ["action_lie_back", "action_lie_side_non_dom", "action_seated_lean_face_down", "action_seated_straight"]:
        bayes_param_spaces[k][f"{pipe_name}__acc_modes"] = Categorical(["raw|velocity|jerk", "smoothed|velocity|displacement|jerk"])
        bayes_param_spaces[k][f"{pipe_name}__rotation_modes"] = Categorical(["quaternion|rot6d|angular_velocity", "delta_euler|angular_velocity"])
        bayes_param_spaces[k][f"{classifier_name}__fusion_mode"] = Categorical(["bigru", "attention"])

    for k in ["position_lie_back", "position_lie_side_non_dom", "position_seated_lean_face_down", "position_seated_straight"]:
        bayes_param_spaces[k][f"{pipe_name}__acc_modes"] = Categorical(["raw|velocity|displacement", "smoothed|velocity|displacement|jerk"])
        bayes_param_spaces[k][f"{pipe_name}__rotation_modes"] = Categorical(["quaternion|rot6d", "quaternion|euler|rot6d|angular_velocity"])
        bayes_param_spaces[k][f"{classifier_name}__fusion_mode"] = Categorical(["attention", "bigru"])

print("Bayesian spaces ready:", list(bayes_param_spaces))

In [ ]:
# ============================================================
# Train / load all 10 models
# ============================================================

search_objects = {}
fitted_models = {}
model_results = {}
model_status = {}
best_params_by_key = {}

for model_key in model_keys:
    target_name = model_target_by_key[model_key]
    family = model_family_by_key[model_key]
    orientation_filter = orientation_filter_by_key[model_key]
    run_mode = model_run_mode[model_key]
    search_mode = model_search_mode[model_key]

    if model_key == "target":
        model_df = train_model_df.copy()
    elif model_key == "orientation":
        model_df = target_only_train_df.copy()
    else:
        model_df = target_only_train_df.loc[
            target_only_train_df["orientation"].eq(orientation_filter)
        ].copy()

    seq_count = model_df["sequence_id"].nunique()
    class_count = model_df.drop_duplicates("sequence_id")[target_name].nunique()

    print("\n" + "=" * 80)
    print("model_key:", model_key)
    print("target:", target_name)
    print("family:", family)
    print("orientation_filter:", orientation_filter)
    print("run_mode:", run_mode)
    print("search_mode:", search_mode)
    print("sequences:", seq_count)
    print("classes:", class_count)

    if seq_count < max(3, n_cv_splits) or class_count < 2:
        print("Skipping model because there are not enough sequences/classes.")
        search_objects[model_key] = None
        fitted_models[model_key] = None
        model_results[model_key] = pd.DataFrame()
        model_status[model_key] = "skipped_insufficient_data"
        best_params_by_key[model_key] = None
        continue

    if family == "cnn":
        estimator = utils.KerasAugmentedCNN1DSequenceClassifier(
            target=target_name,
            verbose=model_verbose,
            random_state=random_state,
        )
    else:
        estimator = hs_utils.DecodingKerasMultiBranchClassifier(
            target=target_name,
            verbose=model_verbose,
            random_state=random_state,
        )

    pipeline = Pipeline([
        (corrector_name, utils.SensorOrientationCorrector(demo_df=train_demo_df)),
        (pipe_name, utils.AdvancedMultiDomainSequenceExtractor()),
        (classifier_name, estimator),
    ])

    y_model = model_df[["sequence_id", target_name]].copy()
    groups_model = model_df["subject"].copy()
    cv_model = GroupKFold(n_splits=n_cv_splits)

    if run_mode == "model":
        if previous_model_paths[model_key] is None:
            print("No model path supplied. Skipping:", model_key)
            search_objects[model_key] = None
            fitted_models[model_key] = None
            model_results[model_key] = pd.DataFrame()
            model_status[model_key] = "skipped_missing_model_path"
            best_params_by_key[model_key] = None
            continue

        loaded_estimator = joblib.load(previous_model_paths[model_key])
        search_obj = SimpleNamespace(
            best_estimator_=loaded_estimator,
            best_params_=getattr(loaded_estimator, "get_params", lambda: {})(),
            best_score_=np.nan,
            cv_results_={"params": ["loaded_model"], "mean_test_score": [np.nan]},
        )
        print("loaded fitted model from:", previous_model_paths[model_key])

    elif run_mode == "best_params":
        if previous_best_params.get(model_key) is None:
            print("No previous params supplied. Skipping:", model_key)
            search_objects[model_key] = None
            fitted_models[model_key] = None
            model_results[model_key] = pd.DataFrame()
            model_status[model_key] = "skipped_missing_best_params"
            best_params_by_key[model_key] = None
            continue

        try:
            pipeline.set_params(**previous_best_params[model_key])
            pipeline.fit(model_df, y_model)
            refit_score = pipeline.score(model_df, y_model)
            search_obj = SimpleNamespace(
                best_estimator_=pipeline,
                best_params_=previous_best_params[model_key],
                best_score_=refit_score,
                cv_results_={"params": [previous_best_params[model_key]], "mean_test_score": [refit_score]},
            )
            print("refit from previous params score:", refit_score)
        except Exception as exc:
            print("Best-params refit failed. Skipping model:", model_key)
            print(type(exc).__name__, exc)
            search_objects[model_key] = None
            fitted_models[model_key] = None
            model_results[model_key] = pd.DataFrame()
            model_status[model_key] = "failed_best_params_refit"
            best_params_by_key[model_key] = None
            continue

    elif run_mode == "search":
        if search_mode == "bayesian":
            if BayesSearchCV is None:
                print("BayesSearchCV unavailable. Skipping:", model_key)
                search_objects[model_key] = None
                fitted_models[model_key] = None
                model_results[model_key] = pd.DataFrame()
                model_status[model_key] = "skipped_bayes_unavailable"
                best_params_by_key[model_key] = None
                continue

            search_obj = BayesSearchCV(
                estimator=pipeline,
                search_spaces=bayes_param_spaces[model_key],
                n_iter=int(model_n_iter[model_key]),
                scoring=None,
                cv=cv_model,
                n_jobs=1,
                refit=True,
                random_state=random_state,
                verbose=search_verbose,
                error_score=error_score,
                return_train_score=True,
            )
        elif search_mode == "grid":
            search_obj = GridSearchCV(
                estimator=pipeline,
                param_grid=grid_param_spaces[model_key],
                scoring=None,
                cv=cv_model,
                n_jobs=1,
                refit=True,
                verbose=search_verbose,
                error_score=error_score,
                return_train_score=True,
            )
        else:
            print("Unknown search mode. Skipping:", search_mode)
            search_objects[model_key] = None
            fitted_models[model_key] = None
            model_results[model_key] = pd.DataFrame()
            model_status[model_key] = "skipped_unknown_search_mode"
            best_params_by_key[model_key] = None
            continue

        try:
            search_obj.fit(model_df, y_model, groups=groups_model)
            print("best score:", search_obj.best_score_)
        except Exception as exc:
            print("Search failed. Skipping model:", model_key)
            print(type(exc).__name__, exc)
            # All fits failed or all candidates were invalid. Keep run alive.
            search_objects[model_key] = None
            fitted_models[model_key] = None
            model_results[model_key] = pd.DataFrame()
            model_status[model_key] = "failed_search"
            best_params_by_key[model_key] = None
            continue
    else:
        print("Unknown run mode. Skipping:", run_mode)
        search_objects[model_key] = None
        fitted_models[model_key] = None
        model_results[model_key] = pd.DataFrame()
        model_status[model_key] = "skipped_unknown_run_mode"
        best_params_by_key[model_key] = None
        continue

    results_df = pd.DataFrame(search_obj.cv_results_)
    cv_path = results_dir / f"cv_results_{model_key}_{timestamp}.csv"
    results_df.to_csv(cv_path, index=False)

    model_path = results_dir / f"fitted_{model_key}_{timestamp}.joblib"
    joblib.dump(search_obj.best_estimator_, model_path)

    search_objects[model_key] = search_obj
    fitted_models[model_key] = search_obj.best_estimator_
    model_results[model_key] = results_df
    model_status[model_key] = "ok"
    best_params_by_key[model_key] = search_obj.best_params_

    print("saved fitted model:", model_path)
    print("saved cv results:", cv_path)
    print("best params:")
    print(search_obj.best_params_)

best_params_path = results_dir / f"orientation_specific_best_params_{timestamp}.json"
with open(best_params_path, "w") as f:
    json.dump(best_params_by_key, f, indent=2, default=str)

status_path = results_dir / f"orientation_specific_model_status_{timestamp}.csv"
pd.DataFrame([{"model_key": k, "status": v} for k, v in model_status.items()]).to_csv(status_path, index=False)

print("\nsaved best params:", best_params_path)
print("saved model status:", status_path)

In [ ]:
# ============================================================
# CV summary so far
# ============================================================

summary_rows = []
for k in model_keys:
    obj = search_objects.get(k)
    summary_rows.append({
        "model_key": k,
        "status": model_status.get(k),
        "target": model_target_by_key[k],
        "family": model_family_by_key[k],
        "orientation_filter": orientation_filter_by_key[k],
        "run_mode": model_run_mode[k],
        "search_mode": model_search_mode[k],
        "best_score": np.nan if obj is None else getattr(obj, "best_score_", np.nan),
    })

cv_summary_df = pd.DataFrame(summary_rows)
cv_summary_path = results_dir / f"orientation_specific_cv_summary_{timestamp}.csv"
cv_summary_df.to_csv(cv_summary_path, index=False)

print(cv_summary_path)
cv_summary_df

In [ ]:
# ============================================================
# Holdout evaluation: route by predicted orientation
# ============================================================

holdout_seq = (
    holdout_df
    .drop_duplicates("sequence_id")
    [["sequence_id", "subject", "sequence_type", "gesture", "hierarchical_gesture", "is_target"]]
    .reset_index(drop=True)
)

holdout_seq.loc[:, "is_target_pred"] = 1
holdout_seq.loc[:, "orientation_pred"] = ""
holdout_seq.loc[:, "gesture_action_pred"] = ""
holdout_seq.loc[:, "gesture_position_pred"] = ""
holdout_seq.loc[:, "reconstructed_gesture_pred"] = fallback_target_gesture
holdout_seq.loc[:, "hierarchical_gesture_pred"] = "Non-Target"

# 1. Target prediction on all holdout sequences
if fitted_models.get("target") is not None:
    pred_target = fitted_models["target"].predict(holdout_df)
    holdout_seq.loc[:, "is_target_pred"] = pred_target.astype(int)
else:
    print("Target model unavailable. Falling back to all predicted Target.")
    holdout_seq.loc[:, "is_target_pred"] = 1

# 2. Orientation prediction on true target holdout rows for head metrics and routing.
# For full hierarchy, this is run on true target holdout only here because final scoring is against labelled validation data.
if fitted_models.get("orientation") is not None and not target_only_holdout_df.empty:
    pred_orientation = fitted_models["orientation"].predict(target_only_holdout_df)
    target_seq_order = target_only_holdout_df.drop_duplicates("sequence_id")["sequence_id"].to_numpy()
    orientation_pred_df = pd.DataFrame({"sequence_id": target_seq_order, "orientation_pred": pred_orientation})
    holdout_seq = holdout_seq.drop(columns=["orientation_pred"]).merge(
        orientation_pred_df,
        on="sequence_id",
        how="left",
    )
else:
    print("Orientation model unavailable. Falling back to true orientation for target rows only.")
    true_orientation_df = target_only_holdout_df.drop_duplicates("sequence_id")[["sequence_id", "orientation"]].rename(columns={"orientation": "orientation_pred"})
    holdout_seq = holdout_seq.drop(columns=["orientation_pred"]).merge(true_orientation_df, on="sequence_id", how="left")

# Attach true target labels for head metrics.
true_target_meta = (
    target_only_holdout_df
    .drop_duplicates("sequence_id")
    [["sequence_id", "orientation", "gesture_action", "gesture_position", "gesture"]]
    .reset_index(drop=True)
)

holdout_seq = holdout_seq.merge(true_target_meta, on="sequence_id", how="left")

# 3. Orientation-specific action / position prediction.
for orientation_value, short_key in orientation_key_map.items():
    seq_ids_for_orientation = holdout_seq.loc[
        holdout_seq["orientation_pred"].eq(orientation_value) & holdout_seq["sequence_type"].eq("Target"),
        "sequence_id",
    ]

    if len(seq_ids_for_orientation) == 0:
        continue

    subset_df = target_only_holdout_df.loc[
        target_only_holdout_df["sequence_id"].isin(seq_ids_for_orientation)
    ].copy()

    action_key = f"action_{short_key}"
    position_key = f"position_{short_key}"

    if fitted_models.get(action_key) is not None:
        action_pred = fitted_models[action_key].predict(subset_df)
        action_order = subset_df.drop_duplicates("sequence_id")["sequence_id"].to_numpy()
        action_pred_map = dict(zip(action_order, action_pred))
    else:
        print("Missing action model, fallback:", action_key)
        action_pred_map = {sid: fallback_action_by_orientation.get(orientation_value, "pull hair") for sid in seq_ids_for_orientation}

    if fitted_models.get(position_key) is not None:
        position_pred = fitted_models[position_key].predict(subset_df)
        position_order = subset_df.drop_duplicates("sequence_id")["sequence_id"].to_numpy()
        position_pred_map = dict(zip(position_order, position_pred))
    else:
        print("Missing position model, fallback:", position_key)
        position_pred_map = {sid: fallback_position_by_orientation.get(orientation_value, "Forehead") for sid in seq_ids_for_orientation}

    holdout_seq.loc[holdout_seq["sequence_id"].isin(action_pred_map.keys()), "gesture_action_pred"] = (
        holdout_seq.loc[holdout_seq["sequence_id"].isin(action_pred_map.keys()), "sequence_id"].map(action_pred_map)
    )

    holdout_seq.loc[holdout_seq["sequence_id"].isin(position_pred_map.keys()), "gesture_position_pred"] = (
        holdout_seq.loc[holdout_seq["sequence_id"].isin(position_pred_map.keys()), "sequence_id"].map(position_pred_map)
    )

# Remaining blanks get orientation-level fallbacks.
target_mask = holdout_seq["sequence_type"].eq("Target")

holdout_seq.loc[target_mask & holdout_seq["gesture_action_pred"].eq(""), "gesture_action_pred"] = (
    holdout_seq.loc[target_mask & holdout_seq["gesture_action_pred"].eq(""), "orientation_pred"]
    .map(fallback_action_by_orientation)
    .fillna("pull hair")
)

holdout_seq.loc[target_mask & holdout_seq["gesture_position_pred"].eq(""), "gesture_position_pred"] = (
    holdout_seq.loc[target_mask & holdout_seq["gesture_position_pred"].eq(""), "orientation_pred"]
    .map(fallback_position_by_orientation)
    .fillna("Forehead")
)

holdout_seq.loc[target_mask, "reconstructed_gesture_pred"] = (
    holdout_seq.loc[target_mask, "gesture_position_pred"].astype(str)
    + " - "
    + holdout_seq.loc[target_mask, "gesture_action_pred"].astype(str)
)

holdout_seq.loc[holdout_seq["is_target_pred"].eq(1), "hierarchical_gesture_pred"] = holdout_seq.loc[
    holdout_seq["is_target_pred"].eq(1),
    "reconstructed_gesture_pred",
]

holdout_seq.loc[holdout_seq["is_target_pred"].eq(0), "hierarchical_gesture_pred"] = "Non-Target"

holdout_seq.head()

In [ ]:
# ============================================================
# Final macro F1 scores + reports
# ============================================================

scores = {}

scores["target_macro_f1"] = f1_score(
    holdout_seq["is_target"],
    holdout_seq["is_target_pred"],
    average="macro",
)

true_target_eval = holdout_seq.loc[holdout_seq["sequence_type"].eq("Target")].copy()

scores["orientation_macro_f1"] = f1_score(
    true_target_eval["orientation"],
    true_target_eval["orientation_pred"],
    average="macro",
)

scores["gesture_action_macro_f1"] = f1_score(
    true_target_eval["gesture_action"],
    true_target_eval["gesture_action_pred"],
    average="macro",
)

scores["gesture_position_macro_f1"] = f1_score(
    true_target_eval["gesture_position"],
    true_target_eval["gesture_position_pred"],
    average="macro",
)

scores["reconstructed_target_gesture_macro_f1"] = f1_score(
    true_target_eval["gesture"],
    true_target_eval["reconstructed_gesture_pred"],
    average="macro",
)

scores["full_hierarchical_macro_f1"] = f1_score(
    holdout_seq["hierarchical_gesture"],
    holdout_seq["hierarchical_gesture_pred"],
    average="macro",
)

scores_df = pd.DataFrame([scores])

print("Final scores")
for k, v in scores.items():
    print(f"{k}: {v:.4f}")

print("\nTarget report")
print(classification_report(holdout_seq["is_target"], holdout_seq["is_target_pred"]))

print("\nOrientation report")
print(classification_report(true_target_eval["orientation"], true_target_eval["orientation_pred"]))

print("\nGesture action report")
print(classification_report(true_target_eval["gesture_action"], true_target_eval["gesture_action_pred"]))

print("\nGesture position report")
print(classification_report(true_target_eval["gesture_position"], true_target_eval["gesture_position_pred"]))

print("\nReconstructed target gesture report")
print(classification_report(true_target_eval["gesture"], true_target_eval["reconstructed_gesture_pred"]))

print("\nFull hierarchy report")
print(classification_report(holdout_seq["hierarchical_gesture"], holdout_seq["hierarchical_gesture_pred"]))

scores_df

In [ ]:
# ============================================================
# Save holdout predictions and summaries
# ============================================================

pred_path = results_dir / f"orientation_specific_holdout_predictions_{timestamp}.csv"
score_path = results_dir / f"orientation_specific_holdout_scores_{timestamp}.csv"

holdout_seq.to_csv(pred_path, index=False)
scores_df.to_csv(score_path, index=False)

print("saved predictions:", pred_path)
print("saved scores:", score_path)
print("saved best params:", best_params_path)
print("saved cv summary:", cv_summary_path)